diptera_estruturas_v3_3.ipynb

Original file is located at
    https://colab.research.google.com/drive/1V9NpBm_V27OWUg6wxx7xq-34x20tEWBW

# Identificacao Hierarquica de Larvas de Mosquito

**Nivel 1:** Genero (Aedes / Anopheles / Culex / Culiseta)

**Nivel 2:** Especie dentro do genero

**Um modelo por estrutura morfologica:** cab, seg8, segAn, sif

Imagens combinadas seg8-segAn sao usadas para treinar ambas as estruturas.

Minimo de **3 larvas** (pastas AutoID) por especie para incluir no treino.

---
Antes de correr: **Runtime > Change runtime type > T4 GPU**

## Imports

Todas as bibliotecas usadas ao longo do notebook sao importadas aqui, no
inicio, em vez de espalhadas por cada passo.

In [ ]:
import os
import json
import time
import shutil
import random
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import classification_report, confusion_matrix

from google.colab import drive
from google.colab import files

random.seed(42)
np.random.seed(42)

print(f'TensorFlow: {tf.__version__}')
print(f'GPU disponivel: {len(tf.config.list_physical_devices("GPU")) > 0}')

## Passo 1 - Ligar ao Google Drive

In [ ]:
# force_remount evita usar uma montagem "presa" de uma sessao anterior,
# uma causa comum do erro "Transport endpoint is not connected".
drive.mount('/content/drive', force_remount=True)
print('Drive ligado.')

## Passo 2 - Configuracao

In [ ]:
# Pasta com as imagens ja redimensionadas (output do pre-processamento)
DATASET_PATH = '/content/drive/MyDrive/Fotos-Micro/Fotos-Micro-512'


def retry_io(fn, *args, max_retries=3, wait_seconds=5, **kwargs):
    """
    Repete uma operacao de I/O sobre o Drive (leitura ou escrita) se falhar
    com um erro transitorio do FUSE do Drive (ex.: "Transport endpoint is
    not connected"). Usado apenas nos pontos especificos onde isso ja
    aconteceu (copia de imagens no Passo 3, gravacao dos modelos), em vez
    de copiar o dataset inteiro para disco local — essa alternativa foi
    tentada e revelou-se, na pratica, muito mais lenta do que aceder ao
    Drive diretamente (o FUSE do Drive tem uma latencia alta por ficheiro,
    e com milhares de imagens isso torna uma copia em bloco impraticavel),
    pelo que foi abandonada a favor deste retry pontual.
    """
    for attempt in range(1, max_retries + 1):
        try:
            return fn(*args, **kwargs)
        except OSError as e:
            if attempt == max_retries:
                raise
            print(f'  Aviso: falha de I/O no Drive ({e}). '
                  f'Tentativa {attempt}/{max_retries}, a repetir em {wait_seconds}s...')
            time.sleep(wait_seconds)

# Estruturas morfologicas
STRUCTURES = ['cab', 'seg8', 'segAn', 'sif']

# Minimo de larvas (pastas AutoID) por especie
MIN_LARVAE = 3

IMG_SIZE   = 224
BATCH_SIZE = 16
EPOCHS_FASE1 = 20  # maximo de epocas da fase 1 (so o classificador novo)
EPOCHS_FASE2 = 10  # maximo de epocas da fase 2 (fine-tuning)
VAL_SPLIT  = 0.2

# Confidence level below which a "low confidence" note is attached to a
# prediction (Passo 10) and used to compute selective-accuracy/coverage
# statistics during evaluation (Passo 8). It does NOT hide or block any
# result — every prediction always reports a confidence % and its
# complementary "unknown %" (100% - confidence).
UNKNOWN_THRESHOLD = 0.55

if os.path.exists(DATASET_PATH):
    folders = sorted(os.listdir(DATASET_PATH))
    print(f'Caminho encontrado: {DATASET_PATH}')
    print(f'{len(folders)} pastas de especies:')
    for f in folders:
        print(f'  {f}')
else:
    print(f'ERRO: caminho nao encontrado: {DATASET_PATH}')

## Passo 3 - Organizar imagens por estrutura, genero e especie

Cria a estrutura de pastas para o TensorFlow:
`/content/datasets/cab/Culex/pipiens/AutoID-xxx_cab.jpg`

Imagens `*-seg8-segAn*` sao copiadas para **ambas** as pastas seg8 e segAn.

Ficheiros com barra de escala (`-sc`) e fotos da lamina sao ignorados.

In [ ]:
BASE_DIR = Path('/content/datasets')
if BASE_DIR.exists():
    shutil.rmtree(BASE_DIR)
BASE_DIR.mkdir()

# Registar larvas por estrutura/genero/especie para contagem correcta
larvae_count = defaultdict(lambda: defaultdict(lambda: defaultdict(set)))

for folder in sorted(Path(DATASET_PATH).iterdir()):
    if not folder.is_dir():
        continue
    name = folder.name
    if '.' not in name:
        continue

    genus, species = name.split('.', 1)

    for larva_folder in sorted(folder.iterdir()):
        if not larva_folder.is_dir():
            continue

        for img in larva_folder.iterdir():
            if img.suffix.lower() not in ['.jpg', '.jpeg', '.png', '.tif', '.tiff']:
                continue
            stem = img.stem.lower()
            if stem.endswith('-sc'):
                continue
            if 'lamina' in stem:
                continue

            # Determinar estrutura(s)
            if '-seg8-segan' in stem:
                targets = ['seg8', 'segAn']
            elif stem.endswith('-cab'):    targets = ['cab']
            elif stem.endswith('-seg8'):   targets = ['seg8']
            elif stem.endswith('-segan'):  targets = ['segAn']
            elif stem.endswith('-sif'):    targets = ['sif']
            else:
                continue

            for structure in targets:
                dest = BASE_DIR / structure / genus / species
                dest.mkdir(parents=True, exist_ok=True)
                retry_io(shutil.copy2, img, dest / f'{larva_folder.name}_{img.name}')
                larvae_count[structure][genus][species].add(larva_folder.name)

# Resumo
print('Larvas por estrutura / genero / especie:')
print('-' * 60)
for struct in STRUCTURES:
    total = sum(len(s) for g in larvae_count[struct].values() for s in g.values())
    print(f'\n{struct.upper()} ({total} larvas total)')
    for genus in sorted(larvae_count[struct]):
        for sp, larva_set in sorted(larvae_count[struct][genus].items()):
            n    = len(larva_set)
            flag = '  [poucos dados]' if n < MIN_LARVAE else ''
            print(f'  {genus}.{sp:<30} {n:>3} larvas{flag}')

## Passo 4 - Remover especies com menos de 3 larvas

In [ ]:
removed = []

for struct in STRUCTURES:
    struct_dir = BASE_DIR / struct
    if not struct_dir.exists():
        continue
    for genus_dir in struct_dir.iterdir():
        if not genus_dir.is_dir():
            continue
        genus = genus_dir.name
        for sp_dir in list(genus_dir.iterdir()):
            species = sp_dir.name
            # Reutiliza a contagem de larvas (pastas AutoID) feita no Passo 3,
            # em vez de reanalisar o nome dos ficheiros aqui (mais fragil e
            # duplicava a logica de deteccao de estrutura/genero/especie).
            n_larvae = len(larvae_count[struct][genus][species])
            if n_larvae < MIN_LARVAE:
                removed.append(f'{struct}/{genus}/{species} ({n_larvae} larvas)')
                shutil.rmtree(sp_dir)
        if not list(genus_dir.iterdir()):
            shutil.rmtree(genus_dir)

if removed:
    print('Removidas (menos de 3 larvas):')
    for r in removed:
        print(f'  {r}')
else:
    print('Todas as especies tem 3 ou mais larvas.')

print('\nDataset final:')
for struct in STRUCTURES:
    struct_dir = BASE_DIR / struct
    if not struct_dir.exists():
        continue
    print(f'  {struct.upper()}')
    for g in sorted(struct_dir.iterdir()):
        sp_list = [s.name for s in sorted(g.iterdir())]
        print(f'    {g.name}: {sp_list}')

## Passo 5 - Definir funcoes

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

# Data augmentation aplicada durante o treino
augment = keras.Sequential([
    keras.layers.RandomFlip('horizontal_and_vertical'),
    keras.layers.RandomRotation(0.3),
    keras.layers.RandomZoom(0.2),
    keras.layers.RandomBrightness(0.2),
    keras.layers.RandomContrast(0.2),
], name='augmentation')


def load_dataset(path, subset):
    """
    Carrega imagens de uma pasta, divide em treino/validacao.

    LIMITACAO CONHECIDA: o split e feito ao nivel da IMAGEM
    (`image_dataset_from_directory` com `validation_split`), nao ao nivel da
    LARVA. Se uma larva tiver mais do que uma imagem da mesma estrutura
    (ex.: fotografias em tentativas ou angulos diferentes — as imagens com
    barra de escala ja foram excluidas no Passo 3), e possivel que caiam
    uma no treino e outra na validacao, o que pode inflacionar ligeiramente
    a accuracy de validacao reportada. O ideal seria dividir por AutoID
    (larva) antes de gerar o dataset por imagem; fica registado como
    limitacao a discutir no relatorio.
    """
    return keras.utils.image_dataset_from_directory(
        path, validation_split=VAL_SPLIT, subset=subset,
        seed=42, image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE)


def build_embedding_model(model):
    """
    Devolve um modelo auxiliar que expoe o "embedding" da larva — o vetor de
    caracteristicas produzido pela CNN antes da camada de classificacao final
    (Dropout e Dense/softmax sao as duas ultimas camadas; o GlobalAveragePooling2D
    e a antepenultima). E este vetor que sera usado para medir se uma imagem
    nova "se parece" com o que o modelo viu no treino.
    """
    return keras.Model(inputs=model.input, outputs=model.layers[-3].output)


def compute_ood_stats(embedding_model, ds, num_classes, safety_margin=1.3):
    """
    Calcula, para cada classe, o centroide (media dos embeddings de treino)
    e uma distancia "de referencia", usada depois para avaliar se uma
    imagem nova esta dentro do que e normal para essa classe.

    A distancia de referencia e a MAIOR distancia observada entre um
    exemplo de TREINO dessa classe e o respetivo centroide, multiplicada
    por uma margem de seguranca (por omissao, 1.3x). Usar o maximo (em vez,
    por exemplo, de um percentil como o 90) garante por construcao que
    QUALQUER exemplo de treino fica sempre a uma distancia igual ou menor
    que a distancia de referencia da sua classe — ao contrario de um
    percentil 90, que classificaria sempre cerca de 10% das proprias
    larvas de treino como "fora do normal", disparando falsos avisos
    mesmo para larvas identicas as usadas no treino (o problema
    observado numa versao anterior desta funcao).

    Para classes com um unico exemplar de treino, a distancia ao proprio
    centroide e sempre 0 (o centroide E o exemplar), pelo que nao e
    possivel estimar variacao normal a partir dos dados dessa classe;
    nesses casos usa-se como aproximacao a mediana das distancias de
    referencia das restantes classes (fallback global).
    """
    embeddings_per_class = defaultdict(list)
    for imgs, labels in ds:
        embs = embedding_model.predict(imgs, verbose=0)
        for e, l in zip(embs, labels.numpy()):
            embeddings_per_class[int(l)].append(e)

    emb_dim = next(iter(embeddings_per_class.values()))[0].shape[0]
    centroids       = np.full((num_classes, emb_dim), np.nan)
    thresholds      = np.zeros(num_classes)
    valid           = np.zeros(num_classes, dtype=bool)
    per_class_thr   = {}

    for c, embs in embeddings_per_class.items():
        embs = np.stack(embs)
        centroid = embs.mean(axis=0)
        centroids[c] = centroid
        valid[c] = True
        if len(embs) >= 2:
            dists = np.linalg.norm(embs - centroid, axis=1)
            per_class_thr[c] = float(dists.max()) * safety_margin
        # classes com 1 unico exemplar ficam sem threshold proprio por agora;
        # preenchidas a seguir com o fallback global

    global_thr = float(np.median(list(per_class_thr.values()))) if per_class_thr else 1.0
    for c in range(num_classes):
        if not valid[c]:
            continue
        thresholds[c] = per_class_thr.get(c, global_thr)

    return centroids, thresholds, valid


def ood_score(embedding, centroids, thresholds, valid):
    """
    Devolve uma percentagem de "desconhecido" (0-100%) para um embedding,
    com base na distancia ao centroide de classe mais proximo, normalizada
    pela distancia de referencia dessa classe (ver compute_ood_stats):

        score = 100 * d / (d + threshold)

    Por construcao do threshold (maximo das distancias de treino x 1.3),
    qualquer larva igual ou semelhante as usadas no treino fica com
    d <= threshold/1.3, logo score < 44% — nunca perto de 100%. Uma larva
    de uma especie totalmente ausente do treino fica, pelo contrario, longe
    de TODOS os centroides, aproximando-se de 100%. Ao contrario de
    "100% - confianca do softmax" (que reparte sempre 100% entre as classes
    conhecidas, mesmo para uma imagem sem qualquer semelhanca com o
    treino), este score reflete a distancia real no espaco de
    caracteristicas da CNN.
    """
    valid_idx = np.where(valid)[0]
    dists = np.linalg.norm(centroids[valid_idx] - embedding, axis=1)
    i = np.argmin(dists)
    d, thr = dists[i], thresholds[valid_idx[i]]
    if thr <= 0:
        return 0.0 if d == 0 else 100.0
    return float(100 * d / (d + thr))


def build_model(num_classes):
    """
    Transfer learning com EfficientNetB0.
    O modelo base ja foi treinado com 1 milhao de imagens (ImageNet)
    e sabe reconhecer formas e texturas gerais. Adicionamos camadas
    finais especificas para classificar larvas de mosquito.
    Treino em 2 fases: (1) so camadas novas, (2) fine-tuning.
    """
    base = keras.applications.EfficientNetB0(
        include_top=False, weights='imagenet',
        input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False

    inputs  = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x       = augment(inputs)
    x       = keras.applications.efficientnet.preprocess_input(x)
    x       = base(x, training=False)
    x       = keras.layers.GlobalAveragePooling2D()(x)
    x       = keras.layers.Dropout(0.3)(x)
    outputs = keras.layers.Dense(num_classes, activation='softmax')(x)

    model = keras.Model(inputs, outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'])
    return model, base


def compute_class_weights(ds):
    """Pesos por classe inversamente proporcionais a frequencia."""
    counts = defaultdict(int)
    for _, labels in ds:
        for label in labels.numpy():
            counts[int(label)] += 1
    total = sum(counts.values())
    n = len(counts)
    return {i: total / (n * counts[i]) for i in counts}


def train_model(model, base, train_ds, val_ds, class_weights=None):
    """Treina em 2 fases: camadas novas e depois fine-tuning."""
    es = keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=5, restore_best_weights=True)
    print('  Fase 1: camadas novas...')
    h1 = model.fit(train_ds, epochs=EPOCHS_FASE1, validation_data=val_ds,
                   class_weight=class_weights, callbacks=[es], verbose=1)

    base.trainable = True
    for layer in base.layers[:-30]:
        layer.trainable = False
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-5),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'])
    es2 = keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=5, restore_best_weights=True)
    print('  Fase 2: fine-tuning...')
    h2 = model.fit(train_ds, epochs=EPOCHS_FASE2, validation_data=val_ds,
                   class_weight=class_weights, callbacks=[es2], verbose=1)

    _, acc = model.evaluate(val_ds, verbose=0)
    print(f'  Accuracy de validacao: {acc * 100:.1f}%')
    return model, [h1, h2]


def plot_history(histories, title):
    acc      = sum([h.history['accuracy']     for h in histories], [])
    val_acc  = sum([h.history['val_accuracy'] for h in histories], [])
    loss     = sum([h.history['loss']         for h in histories], [])
    val_loss = sum([h.history['val_loss']     for h in histories], [])
    split    = len(histories[0].history['accuracy'])
    fig, ax  = plt.subplots(1, 2, figsize=(13, 4))
    fig.suptitle(title, fontweight='bold')
    ax[0].plot(acc, label='Treino'); ax[0].plot(val_acc, label='Validacao')
    ax[0].axvline(split-1, color='gray', linestyle='--', alpha=0.5, label='Fine-tuning')
    ax[0].set_title('Accuracy'); ax[0].set_xlabel('Epoca'); ax[0].legend()
    ax[1].plot(loss, label='Treino'); ax[1].plot(val_loss, label='Validacao')
    ax[1].axvline(split-1, color='gray', linestyle='--', alpha=0.5)
    ax[1].set_title('Loss'); ax[1].set_xlabel('Epoca'); ax[1].legend()
    plt.tight_layout(); plt.show()


def evaluate_model(model, val_ds, class_names, title, unknown_threshold=UNKNOWN_THRESHOLD):
    """
    Avalia o modelo e devolve um dicionario de metricas (para permitir
    construir depois tabelas/graficos-resumo comparando todas as estruturas).
    Mostra a matriz de confusao em contagens e em percentagem (normalizada
    por linha), o que facilita comparar estruturas com suportes diferentes.
    Reporta tambem, para o limiar definido em UNKNOWN_THRESHOLD, que fraccao
    das amostras teria confianca suficiente para nao ser classificada como
    'Desconhecido', e qual a accuracy nessa fraccao (accuracy seletiva).
    """
    y_true, y_pred, confidences = [], [], []
    for imgs, labels in val_ds:
        preds = model.predict(imgs, verbose=0)
        y_pred.extend(np.argmax(preds, axis=1))
        confidences.extend(np.max(preds, axis=1))
        y_true.extend(labels.numpy())
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    confidences = np.array(confidences)

    present = sorted(set(y_true.tolist()))
    names   = [class_names[i] for i in present]

    report = classification_report(
        y_true, y_pred, labels=present, target_names=names,
        output_dict=True, zero_division=0)
    print(f'\n{title}')
    print(classification_report(y_true, y_pred, labels=present, target_names=names, zero_division=0))

    # Nota: quando alguma classe prevista nao tem exemplos verdadeiros neste
    # conjunto de validacao (situacao comum quando ha poucas amostras por
    # especie), o classification_report do sklearn devolve a chave
    # 'micro avg' em vez de 'accuracy'. Para nao depender dessa ambiguidade,
    # a accuracy e calculada diretamente a partir de todas as previsoes.
    accuracy = float((y_pred == y_true).mean())

    coverage = float((confidences >= unknown_threshold).mean())
    if coverage > 0:
        mask = confidences >= unknown_threshold
        sel_acc = float((y_pred[mask] == y_true[mask]).mean())
    else:
        sel_acc = float('nan')
    print(f'  Ao limiar de {unknown_threshold*100:.0f}%: '
          f'{coverage*100:.1f}% das amostras seriam classificadas '
          f'(restantes cairiam em "Desconhecido"); '
          f'accuracy nessas amostras classificadas: {sel_acc*100:.1f}%')

    cm      = confusion_matrix(y_true, y_pred, labels=present)
    cm_norm = confusion_matrix(y_true, y_pred, labels=present, normalize='true')

    fig, axes = plt.subplots(1, 2, figsize=(max(10, len(names)*2.0), max(4.5, len(names)*0.9)))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=names, yticklabels=names, ax=axes[0])
    axes[0].set_title(f'Contagens - {title}')
    axes[0].set_ylabel('Classe real'); axes[0].set_xlabel('Classe prevista')
    axes[0].tick_params(axis='x', rotation=45)

    sns.heatmap(cm_norm, annot=True, fmt='.0%', cmap='Blues', vmin=0, vmax=1,
                xticklabels=names, yticklabels=names, ax=axes[1])
    axes[1].set_title(f'Normalizada por linha (%) - {title}')
    axes[1].set_ylabel('Classe real'); axes[1].set_xlabel('Classe prevista')
    axes[1].tick_params(axis='x', rotation=45)
    plt.tight_layout(); plt.show()

    return {
        'title': title,
        'accuracy': accuracy,
        'macro_f1': report['macro avg']['f1-score'],
        'weighted_f1': report['weighted avg']['f1-score'],
        'coverage_at_threshold': coverage,
        'selective_accuracy': sel_acc,
        'n': int(len(y_true)),
    }


def plot_history_grid(items, suptitle):
    """
    Desenha, numa unica figura (small multiples), a curva de accuracy de
    treino/validacao de cada item em 'items' (lista de (nome, histories)).
    Substitui varias figuras separadas por uma so, facilitando a comparacao
    e reduzindo o numero de imagens a incluir no relatorio.
    """
    n = len(items)
    if n == 0:
        return
    fig, axes = plt.subplots(1, n, figsize=(5*n, 4), sharey=True)
    if n == 1:
        axes = [axes]
    for ax, (name, histories) in zip(axes, items):
        acc     = sum([h.history['accuracy']     for h in histories], [])
        val_acc = sum([h.history['val_accuracy'] for h in histories], [])
        split   = len(histories[0].history['accuracy'])
        ax.plot(acc, label='Treino'); ax.plot(val_acc, label='Validacao')
        ax.axvline(split-1, color='gray', linestyle='--', alpha=0.5)
        ax.set_title(name); ax.set_xlabel('Epoca'); ax.set_ylim(0, 1)
    axes[0].set_ylabel('Accuracy'); axes[0].legend()
    fig.suptitle(suptitle, fontweight='bold')
    plt.tight_layout(); plt.show()


print('Funcoes definidas.')

## Passo 6 - Treinar modelos Nivel 1: Genero

Para cada estrutura, um modelo que distingue os 4 generos.

In [ ]:
models_genus = {}

SAVE_DIR = '/content/drive/MyDrive/diptera_models'
os.makedirs(SAVE_DIR, exist_ok=True)

for struct in STRUCTURES:
    struct_dir = BASE_DIR / struct
    if not struct_dir.exists():
        print(f'\n{struct.upper()}: sem dados, ignorado.')
        continue
    genera = [d.name for d in struct_dir.iterdir() if d.is_dir()]
    if len(genera) < 2:
        print(f'\n{struct.upper()}: so 1 genero, ignorado.')
        continue

    print(f'\n{"="*55}')
    print(f'GENERO - {struct.upper()} ({len(genera)} generos): {genera}')
    print(f'{"="*55}')

    # Load dataset first to get class_names
    raw_train_ds = load_dataset(struct_dir, 'training')
    raw_val_ds   = load_dataset(struct_dir, 'validation')

    class_names   = raw_train_ds.class_names

    train_ds = raw_train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
    val_ds   = raw_val_ds.cache().prefetch(AUTOTUNE)

    class_weights = compute_class_weights(train_ds)
    model, base   = build_model(len(class_names))
    model, histories = train_model(model, base, train_ds, val_ds, class_weights)
    models_genus[struct] = (model, class_names, histories)

    # Guardar imediatamente: protege o trabalho ja feito caso o runtime do
    # Colab se desligue a meio do ciclo seguinte (ex: modelo de outra estrutura).
    retry_io(model.save, f'{SAVE_DIR}/genus_{struct}.keras')
    print(f'  Guardado (parcial): genus_{struct}.keras')

    # Estatisticas para deteccao de "desconhecido" (distancia a classes conhecidas)
    embedding_model = build_embedding_model(model)
    centroids, thresholds, valid = compute_ood_stats(embedding_model, train_ds, len(class_names))
    retry_io(np.savez, f'{SAVE_DIR}/genus_{struct}_ood.npz',
             centroids=centroids, thresholds=thresholds, valid=valid)
    print(f'  Guardado (parcial): genus_{struct}_ood.npz')

print('\nTodos os modelos de genero treinados.')

## Passo 7 - Treinar modelos Nivel 2: Especie

Para cada estrutura e cada genero, um modelo de especie.

*Anopheles* nao tem sifao — ignorado automaticamente.

In [ ]:
models_species = defaultdict(dict)
meta_partial = {'genus': {s: cn.tolist() if hasattr(cn, 'tolist') else list(cn)
                           for s, (_, cn, _) in models_genus.items()},
                 'species': defaultdict(dict)}

for struct in STRUCTURES:
    struct_dir = BASE_DIR / struct
    if not struct_dir.exists():
        continue
    for genus_dir in sorted(struct_dir.iterdir()):
        if not genus_dir.is_dir():
            continue
        genus        = genus_dir.name
        species_list = [d.name for d in genus_dir.iterdir() if d.is_dir()]
        if len(species_list) < 2:
            print(f'{struct.upper()} / {genus}: so 1 especie, ignorado.')
            continue

        print(f'\n{"="*55}')
        print(f'ESPECIE - {struct.upper()} / {genus} ({len(species_list)} especies)')
        print(f'Especies: {species_list}')
        print(f'{"="*55}')

        # Load dataset first to get class_names
        raw_train_ds = load_dataset(genus_dir, 'training')
        raw_val_ds   = load_dataset(genus_dir, 'validation')

        class_names = raw_train_ds.class_names

        train_ds = raw_train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
        val_ds   = raw_val_ds.cache().prefetch(AUTOTUNE)

        class_weights = compute_class_weights(train_ds)
        model, base   = build_model(len(species_list))
        model, histories = train_model(model, base, train_ds, val_ds, class_weights)
        models_species[struct][genus] = (model, class_names, histories)

        # Guardar imediatamente (modelo + metadados), para nao perder trabalho
        # se o runtime desligar antes de chegar ao Passo 9.
        retry_io(model.save, f'{SAVE_DIR}/species_{struct}_{genus}.keras')
        meta_partial['species'][struct][genus] = list(class_names)

        def _save_classes_json():
            with open(f'{SAVE_DIR}/classes.json', 'w') as f:
                json.dump(meta_partial, f, indent=2)

        retry_io(_save_classes_json)
        print(f'  Guardado (parcial): species_{struct}_{genus}.keras')

        embedding_model = build_embedding_model(model)
        centroids, thresholds, valid = compute_ood_stats(embedding_model, train_ds, len(class_names))
        retry_io(np.savez, f'{SAVE_DIR}/species_{struct}_{genus}_ood.npz',
                 centroids=centroids, thresholds=thresholds, valid=valid)
        print(f'  Guardado (parcial): species_{struct}_{genus}_ood.npz')

print('\nTodos os modelos de especie treinados.')

## Passo 8 - Avaliar todos os modelos

In [ ]:
# Curvas de accuracy dos 4 modelos de genero numa unica figura (em vez de 4)
plot_history_grid(
    [(struct.upper(), hist) for struct, (_, _, hist) in sorted(models_genus.items())],
    'Accuracy - Modelos de Genero (todas as estruturas)')

results_genus = []
for struct, (model, class_names, histories) in sorted(models_genus.items()):
    val_ds = load_dataset(BASE_DIR / struct, 'validation').cache().prefetch(AUTOTUNE)
    r = evaluate_model(model, val_ds, class_names, f'Genero - {struct.upper()}')
    r['structure'] = struct
    results_genus.append(r)

results_species = []
for struct in sorted(models_species):
    # Curvas de accuracy dos generos desta estrutura, numa unica figura
    plot_history_grid(
        [(genus, hist) for genus, (_, _, hist) in sorted(models_species[struct].items())],
        f'Accuracy - Modelos de Especie - {struct.upper()} (todos os generos)')

    for genus, (model, class_names, histories) in sorted(models_species[struct].items()):
        val_ds = load_dataset(BASE_DIR / struct / genus, 'validation').cache().prefetch(AUTOTUNE)
        r = evaluate_model(model, val_ds, class_names, f'Especie - {struct.upper()} / {genus}')
        r['structure'] = struct
        r['genus'] = genus
        results_species.append(r)

df_genus   = pd.DataFrame(results_genus)
df_species = pd.DataFrame(results_species)

print('\nResumo - Modelos de Genero')
print(df_genus[['structure', 'accuracy', 'macro_f1', 'weighted_f1', 'n']].to_string(index=False))

print('\nResumo - Modelos de Especie')
print(df_species[['structure', 'genus', 'accuracy', 'macro_f1', 'weighted_f1', 'n']].to_string(index=False))

# Guardar em CSV para facilitar a construcao das tabelas do relatorio
retry_io(df_genus.to_csv, f'{SAVE_DIR}/resumo_genero.csv', index=False)
retry_io(df_species.to_csv, f'{SAVE_DIR}/resumo_especie.csv', index=False)
print(f'\nTabelas resumo guardadas em {SAVE_DIR}/resumo_genero.csv e resumo_especie.csv')

## Passo 8b - Visualizacoes resumo (comparar estruturas de uma so vez)

Em vez de olhar para 19 conjuntos de graficos separados (repetitivo e dificil
de comparar), esta seccao produz um pequeno numero de figuras agregadas,
pensadas para ir para o corpo do relatorio (as figuras individuais do Passo 8
podem ficar em anexo, mantendo aqui apenas 1-2 exemplos ilustrativos).

In [ ]:
# --- Heatmap resumo: Genero (estrutura x metrica) ---
pivot_genus = df_genus.set_index('structure')[['accuracy', 'macro_f1', 'weighted_f1']]
plt.figure(figsize=(6, 0.6*len(pivot_genus) + 1.5))
sns.heatmap(pivot_genus, annot=True, fmt='.2f', cmap='Blues', vmin=0, vmax=1)
plt.title('Resumo - Modelos de Genero (por estrutura)')
plt.tight_layout(); plt.show()

# --- Heatmap resumo: Especie (estrutura x genero), accuracy e macro F1 ---
for metric, label in [('accuracy', 'Accuracy'), ('macro_f1', 'Macro F1')]:
    pivot = df_species.pivot(index='structure', columns='genus', values=metric)
    plt.figure(figsize=(6, 0.6*len(pivot) + 1.5))
    sns.heatmap(pivot, annot=True, fmt='.2f', cmap='Blues', vmin=0, vmax=1)
    plt.title(f'Resumo - {label} dos Modelos de Especie (estrutura x genero)')
    plt.tight_layout(); plt.show()

# --- Grafico de barras: accuracy dos modelos de genero, por estrutura ---
plt.figure(figsize=(6, 4))
sns.barplot(data=df_genus, x='structure', y='accuracy', color='#4C72B0')
plt.ylim(0, 1); plt.title('Accuracy - Modelos de Genero, por estrutura')
plt.ylabel('Accuracy'); plt.xlabel('Estrutura')
plt.tight_layout(); plt.show()

# --- Grafico de barras: macro F1 dos modelos de especie, por genero e estrutura ---
plt.figure(figsize=(8, 4.5))
sns.barplot(data=df_species, x='genus', y='macro_f1', hue='structure')
plt.ylim(0, 1); plt.title('Macro F1 - Modelos de Especie, por genero e estrutura')
plt.ylabel('Macro F1'); plt.xlabel('Genero')
plt.legend(title='Estrutura', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout(); plt.show()


def evaluate_pooled_species(struct):
    """
    Junta, numa unica matriz de confusao, os resultados de especie de TODOS
    os generos para uma dada estrutura (em vez de uma matriz por genero).
    NOTA: como cada modelo de especie so ve imagens do seu proprio genero,
    esta matriz e necessariamente em blocos (nao ha, nem pode haver, erros
    entre especies de generos diferentes nesta avaliacao) — o objetivo e
    apenas reduzir o numero de figuras a apresentar, nao avaliar confusao
    entre generos (para isso, ver a avaliacao do pipeline completo).
    """
    y_true_all, y_pred_all, label_names = [], [], []
    offset = 0
    for genus, (model, class_names, _) in sorted(models_species.get(struct, {}).items()):
        val_ds = load_dataset(BASE_DIR / struct / genus, 'validation').cache().prefetch(AUTOTUNE)
        y_true_g, y_pred_g = [], []
        for imgs, labels in val_ds:
            preds = model.predict(imgs, verbose=0)
            y_pred_g.extend(np.argmax(preds, axis=1))
            y_true_g.extend(labels.numpy())
        y_true_all.extend(offset + t for t in y_true_g)
        y_pred_all.extend(offset + p for p in y_pred_g)
        label_names.extend(f'{genus}.{c}' for c in class_names)
        offset += len(class_names)

    if not label_names:
        return
    cm = confusion_matrix(y_true_all, y_pred_all, labels=list(range(len(label_names))))
    plt.figure(figsize=(max(8, len(label_names)*0.6), max(6, len(label_names)*0.55)))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=label_names, yticklabels=label_names)
    plt.title(f'Matriz de Confusao agregada - Especie - {struct.upper()} (todos os generos)')
    plt.ylabel('Classe real (genero.especie)'); plt.xlabel('Classe prevista (genero.especie)')
    plt.xticks(rotation=90); plt.yticks(rotation=0)
    plt.tight_layout(); plt.show()


for struct in STRUCTURES:
    evaluate_pooled_species(struct)

## Passo 8c - Resumo do treino (epocas por fase e metricas finais)

Celula independente do Passo 8/8b — nao altera nenhum grafico ou resultado
ja obtido, e nao e necessario retreinar nada para a correr (usa os
`histories` ja guardados em memoria em `models_genus` / `models_species`
depois dos Passos 6 e 7). Serve apenas para documentar, por modelo, quantas
epocas cada fase de treino efetivamente usou (pode ser menos do que o
maximo definido em EPOCHS_FASE1/EPOCHS_FASE2, por causa do early stopping)
e a accuracy/loss de validacao no final do treino.

Com 19 modelos, esta tabela sai com 19 linhas — considerar colocar em
anexo no relatorio, e no corpo do texto referir apenas o intervalo
(minimo-maximo) de epocas observado e um ou dois exemplos ilustrativos.

In [ ]:
def summarize_training(histories):
    """
    A partir da lista de historicos de treino (um por fase), devolve
    quantas epocas cada fase efetivamente correu e a accuracy/loss de
    validacao no final do treino (ja com os melhores pesos restaurados
    pelo early stopping).
    """
    epochs_per_phase = [len(h.history['accuracy']) for h in histories]
    return {
        'epochs_fase1': epochs_per_phase[0] if len(epochs_per_phase) > 0 else None,
        'epochs_fase2': epochs_per_phase[1] if len(epochs_per_phase) > 1 else None,
        'val_accuracy_final': float(histories[-1].history['val_accuracy'][-1]),
        'val_loss_final': float(histories[-1].history['val_loss'][-1]),
    }


rows_treino = []
for struct, (_, _, histories) in sorted(models_genus.items()):
    row = {'modelo': f'Genero - {struct.upper()}'}
    row.update(summarize_training(histories))
    rows_treino.append(row)

for struct in sorted(models_species):
    for genus, (_, _, histories) in sorted(models_species[struct].items()):
        row = {'modelo': f'Especie - {struct.upper()} / {genus}'}
        row.update(summarize_training(histories))
        rows_treino.append(row)

df_treino = pd.DataFrame(rows_treino)
print(df_treino.to_string(index=False))
print(f'\nEpocas Fase 1 usadas: min={df_treino.epochs_fase1.min()}, max={df_treino.epochs_fase1.max()} '
      f'(maximo permitido: {EPOCHS_FASE1})')
print(f'Epocas Fase 2 usadas: min={df_treino.epochs_fase2.min()}, max={df_treino.epochs_fase2.max()} '
      f'(maximo permitido: {EPOCHS_FASE2})')

retry_io(df_treino.to_csv, f'{SAVE_DIR}/resumo_treino.csv', index=False)
print(f'\nGuardado em {SAVE_DIR}/resumo_treino.csv')

## Passo 9 - Confirmar que tudo ficou guardado

Os modelos e o `classes.json` ja foram guardados incrementalmente nos Passos 6 e 7
(protecao contra desconexoes do runtime). Este passo e apenas uma confirmacao final.

In [ ]:
print('Modelos de genero guardados:')
for struct in models_genus:
    print(f'  genus_{struct}.keras + genus_{struct}_ood.npz')

print('\nModelos de especie guardados:')
for struct in models_species:
    for genus in models_species[struct]:
        print(f'  species_{struct}_{genus}.keras + species_{struct}_{genus}_ood.npz')

print(f'\nMetadados em: {SAVE_DIR}/classes.json')

## Passo 10 - Testar com imagens novas

Carrega fotos de uma larva. O sistema combina as previsoes por votacao ponderada.

A percentagem de "desconhecido" reportada aqui NAO e "100% - confianca do
softmax" (essa formula reparte sempre 100% pelas classes conhecidas, mesmo
para uma larva completamente fora do dataset de treino). Em vez disso, mede a
distancia do "embedding" da imagem (o vetor de caracteristicas da CNN) ao
centroide da classe conhecida mais proxima, normalizada pela variacao maxima
observada dentro dessa classe no treino: uma larva de uma especie nunca vista
fica longe de TODOS os centroides (desconhecido alto), enquanto uma larva de
uma especie conhecida fica, por construcao, sempre com um desconhecido baixo.

O desconhecido e apresentado como mais uma "fatia" do voto combinado: as
percentagens das classes conhecidas e a percentagem de desconhecido somam
sempre 100%.

Nome dos ficheiros deve incluir a estrutura: `larva1-cab.jpg`, `larva1-sif.jpg`, etc.

In [ ]:
print('Carrega as fotos da larva (podes carregar varias ao mesmo tempo).')
uploaded = files.upload()


def load_image_array(img_path):
    img = keras.utils.load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
    return tf.expand_dims(keras.utils.img_to_array(img), 0)


def load_ood_stats(path):
    data = np.load(path)
    return data['centroids'], data['thresholds'], data['valid']


def print_vote_with_unknown(votes_pct, unknown_pct, bar_width=30):
    """
    Imprime a votacao combinada como percentagens que somam 100% (as classes
    conhecidas, rescaladas por (100 - unknown_pct)/100, mais a fatia de
    desconhecido), com uma barra proporcional para cada uma.
    """
    for name, pct in sorted(votes_pct.items(), key=lambda x: -x[1]):
        bar = '|' * int(round(pct / 100 * bar_width))
        print(f'    {name:<16} {pct:5.1f}%  {bar}')
    bar = '|' * int(round(unknown_pct / 100 * bar_width))
    print(f'    {"Desconhecido":<16} {unknown_pct:5.1f}%  {bar}')


file_by_struct = defaultdict(list)
for fname in uploaded:
    fl = fname.lower()
    if fl.endswith('-sc.jpg'):
        continue
    if 'cab'   in fl:  file_by_struct['cab'].append(fname)
    elif 'seg8'  in fl: file_by_struct['seg8'].append(fname)
    elif 'segan' in fl: file_by_struct['segAn'].append(fname)
    elif 'sif'   in fl: file_by_struct['sif'].append(fname)

print('\nFicheiros por estrutura:')
for s, flist in file_by_struct.items():
    print(f'  {s}: {flist}')

# Nivel 1: Genero
print('\n' + '-'*50)
print('Nivel 1 - Genero (votacao ponderada)')
print('-'*50)
genus_votes = defaultdict(float)
genus_ood_scores = []

for struct, fnames in file_by_struct.items():
    if struct not in models_genus:
        continue
    model, class_names, _ = models_genus[struct]
    emb_model = build_embedding_model(model)
    centroids, thresholds, valid = load_ood_stats(f'{SAVE_DIR}/genus_{struct}_ood.npz')
    for fname in fnames:
        arr   = load_image_array(fname)
        probs = model.predict(arr, verbose=0)[0]
        emb   = emb_model.predict(arr, verbose=0)[0]
        score = ood_score(emb, centroids, thresholds, valid)
        genus_ood_scores.append(score)
        for i, p in enumerate(probs):
            genus_votes[class_names[i]] += p
        print(f'  {struct}: {", ".join(f"{class_names[i]}: {probs[i]*100:.1f}%" for i in range(len(class_names)))}')

if genus_votes:
    total_v         = sum(genus_votes.values())
    genus_unknown   = float(np.mean(genus_ood_scores)) if genus_ood_scores else 0.0
    known_share     = 100 - genus_unknown
    genus_pct       = {g: (v / total_v) * known_share for g, v in genus_votes.items()}
    predicted_genus = max(genus_pct, key=genus_pct.get)

    print('\n  Votacao combinada:')
    print_vote_with_unknown(genus_pct, genus_unknown)
    print(f'\n  Genero mais provavel: {predicted_genus} ({genus_pct[predicted_genus]:.1f}%)')

    # Nivel 2: Especie (calculada sempre, mesmo com desconhecido elevado no genero)
    print('\n' + '-'*50)
    print(f'Nivel 2 - Especie dentro de {predicted_genus}')
    print('-'*50)
    species_votes = defaultdict(float)
    species_ood_scores = []

    for struct, fnames in file_by_struct.items():
        if struct not in models_species:
            continue
        if predicted_genus not in models_species[struct]:
            continue
        model, sp_names, _ = models_species[struct][predicted_genus]
        emb_model = build_embedding_model(model)
        ood_path = f'{SAVE_DIR}/species_{struct}_{predicted_genus}_ood.npz'
        centroids, thresholds, valid = load_ood_stats(ood_path)
        for fname in fnames:
            arr   = load_image_array(fname)
            probs = model.predict(arr, verbose=0)[0]
            emb   = emb_model.predict(arr, verbose=0)[0]
            score = ood_score(emb, centroids, thresholds, valid)
            species_ood_scores.append(score)
            for i, p in enumerate(probs):
                species_votes[sp_names[i]] += p
            print(f'  {struct}: {", ".join(f"{sp_names[i]}: {probs[i]*100:.1f}%" for i in range(len(sp_names)))}')

    if species_votes:
        total_sv           = sum(species_votes.values())
        species_unknown     = float(np.mean(species_ood_scores)) if species_ood_scores else 0.0
        sp_known_share      = 100 - species_unknown
        species_pct         = {s: (v / total_sv) * sp_known_share for s, v in species_votes.items()}
        predicted_species   = max(species_pct, key=species_pct.get)

        print('\n  Votacao combinada:')
        print_vote_with_unknown(species_pct, species_unknown)
        print(f'\n  Especie mais provavel: {predicted_species} ({species_pct[predicted_species]:.1f}%)')

        print(f'\n  Identificacao final: {predicted_genus} {predicted_species}')
        print(f'    Genero:  {genus_pct[predicted_genus]:.1f}%  ({genus_unknown:.1f}% desconhecido)')
        print(f'    Especie: {species_pct[predicted_species]:.1f}%  ({species_unknown:.1f}% desconhecido)')
    else:
        print(f'  Sem modelo de especie para {predicted_genus}.')